# Feature Engineering — adapted for `matchups.csv` (T1/T2 wide format)

This is the original `build_features` notebook adapted to read the **T1/T2 wide-format** `matchups.csv` (one row per game with `T1_*` / `T2_*` columns side-by-side) instead of the team-perspective doubled format the original was written for.

### What changed vs. the original

**Step 0 (new) — Reshape.** A new prep step converts the wide format into the doubled team-perspective format (each game → 2 rows, one per team's view) and computes `Diff_*` columns from every `T1_*` / `T2_*` pair. After this step, the dataframe matches what the original notebook expected and the rest of the pipeline runs unchanged.

**Home/away assumption.** Your `matchups.csv` has no explicit home flag. The T1 win rate in your data is ~0.562 — right in line with NBA home-court win rate — so this notebook assumes **T1 is the home team**. If your dataset builder uses a different convention, change the `T1_IS_HOME` flag in the config cell.

### Features added (same as the original)

**Diffs (this team − opponent):**
- `Diff_Days_Rest`
- `Diff_Win_Streak`
- `Diff_Last5_Win_Pct`, `Diff_Last10_Win_Pct`

**Per-team:**
- `Team_Is_B2B`, `Opp_Is_B2B` — back-to-back flags
- `H2H_Win_Pct` — this team's win pct vs `OppID` in prior matchups this season
- `Elo_Win_Prob` — implied win probability from `Diff_Pregame_Elo` with home-court bump

All features use only games **strictly before** each game (in the same season).

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Config ----
INPUT_CSV    = "matchups.csv"
OUTPUT_CSV   = "matchups_v2.csv"
ELO_HOME_ADV = 100.0   # match the constant your dataset builder used
T1_IS_HOME   = True    # In your data, T1 win rate ~0.562 ≈ NBA home win rate, so T1 = home.
                       # Set to False if your convention is different (or set per-row if you have it).
OUT_DIR      = Path(".")
# ----------------

wide = pd.read_csv(INPUT_CSV, parse_dates=['DayDate'])
print(f"{INPUT_CSV}: {wide.shape}")
print(f"Seasons: {sorted(wide['Season'].unique())}")
print(f"T1 win rate: {wide['Target_Win'].mean():.3f}  (≈ NBA home-court rate ~ 0.56–0.58)")

matchups.csv: (11969, 83)
Seasons: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
T1 win rate: 0.562  (≈ NBA home-court rate ~ 0.56–0.58)


## Step 0 — Reshape T1/T2 wide → team-perspective long

Each game in the wide CSV becomes **two rows** in the long format:
- One row from T1's perspective (`TeamID = T1_TeamID`, `OppID = T2_TeamID`, `Target_Win` unchanged)
- One row from T2's perspective (`TeamID = T2_TeamID`, `OppID = T1_TeamID`, `Target_Win = 1 − original`)

We rename `T1_*` → `team_*` (or `opp_*` depending on perspective) and compute `Diff_<stat> = team_<stat> − opp_<stat>` for every paired stat column. After this cell, `mm` has the same schema the original `build_features` notebook expected.

In [2]:
def wide_to_team_perspective(wide: pd.DataFrame, t1_is_home: bool = True) -> pd.DataFrame:
    """Reshape a T1/T2 wide-format game frame into a team-perspective long frame.

    Each input row produces two output rows. Returns a frame with:
      - shared keys: Season, DayDate, GameID, ID
      - TeamID, OppID, Target_Win, Is_Home
      - team_<stat>, opp_<stat>, Diff_<stat>  for every T1/T2 paired column
    """
    t1_cols = [c for c in wide.columns if c.startswith('T1_')]
    t2_cols = [c for c in wide.columns if c.startswith('T2_')]

    # Stat names that exist on both sides (e.g. 'TeamID', 'Pregame_Elo', 'Avg_Score', ...)
    t1_stats = {c[len('T1_'):] for c in t1_cols}
    t2_stats = {c[len('T2_'):] for c in t2_cols}
    paired_stats = sorted(t1_stats & t2_stats)
    only_t1 = sorted(t1_stats - t2_stats)
    only_t2 = sorted(t2_stats - t1_stats)
    if only_t1 or only_t2:
        print(f"  warning: unpaired columns -- T1 only: {only_t1} | T2 only: {only_t2}")

    shared_cols = [c for c in wide.columns
                   if not c.startswith('T1_') and not c.startswith('T2_') and c != 'Target_Win']

    def build_view(team_prefix: str, opp_prefix: str, target_win: pd.Series, is_home: int) -> pd.DataFrame:
        rename = {}
        for s in paired_stats:
            rename[f'{team_prefix}{s}'] = f'team_{s}'
            rename[f'{opp_prefix}{s}']  = f'opp_{s}'
        view = wide.rename(columns=rename)[shared_cols
                                          + [f'team_{s}' for s in paired_stats]
                                          + [f'opp_{s}'  for s in paired_stats]].copy()
        view['Target_Win'] = target_win.values
        view['Is_Home']    = is_home
        return view

    t1_view = build_view('T1_', 'T2_',
                         target_win=wide['Target_Win'],
                         is_home=int(t1_is_home))
    t2_view = build_view('T2_', 'T1_',
                         target_win=1 - wide['Target_Win'],
                         is_home=int(not t1_is_home))

    long = pd.concat([t1_view, t2_view], ignore_index=True)

    # Promote TeamID / OppID to canonical names (the original notebook uses these)
    long = long.rename(columns={'team_TeamID': 'TeamID', 'opp_TeamID': 'OppID'})

    # Compute Diff_<stat> = team_<stat> - opp_<stat> for every numeric paired column
    for s in paired_stats:
        if s == 'TeamID':
            continue
        team_c, opp_c = f'team_{s}', f'opp_{s}'
        if pd.api.types.is_numeric_dtype(long[team_c]) and pd.api.types.is_numeric_dtype(long[opp_c]):
            long[f'Diff_{s}'] = long[team_c] - long[opp_c]

    return long


mm = wide_to_team_perspective(wide, t1_is_home=T1_IS_HOME)
print(f"Reshaped: {wide.shape} -> {mm.shape}  (rows doubled: {len(mm) == 2*len(wide)})")
print(f"\nKey columns present: "
      f"{[c for c in ['TeamID','OppID','Target_Win','Is_Home','Diff_Pregame_Elo','Diff_Avg_Score'] if c in mm.columns]}")
print(f"\nFirst 4 rows (showing both perspectives of the first 2 games):")
mm.sort_values(['GameID','TeamID']).head(4)[['Season','DayDate','GameID','TeamID','OppID',
                                              'Is_Home','Target_Win','Diff_Pregame_Elo']]

Reshaped: (11969, 83) -> (23938, 122)  (rows doubled: True)

Key columns present: ['TeamID', 'OppID', 'Target_Win', 'Is_Home', 'Diff_Pregame_Elo', 'Diff_Avg_Score']

First 4 rows (showing both perspectives of the first 2 games):


,Season,DayDate,GameID,TeamID,OppID,Is_Home,Target_Win,Diff_Pregame_Elo
0,2016,2016-10-25,21600001,1610612739,1610612752,1,1,0.0
11969,2016,2016-10-25,21600001,1610612752,1610612739,0,0,0.0
1,2016,2016-10-25,21600002,1610612757,1610612762,1,1,0.0
11970,2016,2016-10-25,21600002,1610612762,1610612757,0,0,0.0


## Step 1 — Per-team chronological features

Sort by `(TeamID, Season, DayDate, GameID)` and walk each team's chronological log. Use `shift(1)` so the current game's outcome (`Target_Win`) never leaks into its own features.

In [3]:
def add_per_team_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['TeamID', 'Season', 'DayDate', 'GameID']).reset_index(drop=True).copy()
    grp_keys = ['TeamID', 'Season']

    # ---- Days_Rest: days since this team's previous game in same season ----
    prev_date = df.groupby(grp_keys)['DayDate'].shift(1)
    days_rest = (df['DayDate'] - prev_date).dt.days
    df['Days_Rest'] = days_rest.clip(upper=14)         # cap absurd gaps
    df['Is_B2B']    = (df['Days_Rest'] == 1).astype('Int64')
    df.loc[df['Days_Rest'].isna(), 'Is_B2B'] = pd.NA

    # ---- Win_Streak: +N for N-game win streak entering this game, -N for losses ----
    def streak(group):
        prev_won = group['Target_Win'].shift(1)
        s = 0; out = []
        for w in prev_won:
            if pd.isna(w):
                out.append(np.nan); s = 0
            elif w == 1:
                s = s + 1 if s >= 0 else 1
                out.append(s)
            else:
                s = s - 1 if s <= 0 else -1
                out.append(s)
        return pd.Series(out, index=group.index)

    df['Win_Streak'] = df.groupby(grp_keys, group_keys=False).apply(streak)

    # ---- Rolling N-game win pct (current game excluded via shift(1)) ----
    def rolling_winpct(n):
        return (df.groupby(grp_keys)['Target_Win']
                  .apply(lambda s: s.shift(1).rolling(n, min_periods=1).mean())
                  .reset_index(level=grp_keys, drop=True))

    df['Last5_Win_Pct']  = rolling_winpct(5)
    df['Last10_Win_Pct'] = rolling_winpct(10)

    return df


mm = add_per_team_features(mm)
mm[['Season','DayDate','TeamID','OppID','Target_Win',
    'Days_Rest','Is_B2B','Win_Streak',
    'Last5_Win_Pct','Last10_Win_Pct']].head(15)

C:\Users\Furag\AppData\Local\Temp\ipykernel_7324\3493191991.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['Win_Streak'] = df.groupby(grp_keys, group_keys=False).apply(streak)


,Season,DayDate,TeamID,OppID,Target_Win,Days_Rest,Is_B2B,Win_Streak,Last5_Win_Pct,Last10_Win_Pct
0,2016,2016-10-27,1610612737,1610612764,1,NaN,<NA>,NaN,NaN,NaN
1,2016,2016-10-29,1610612737,1610612755,1,2.0,0,1.0,1.00,1.000000
2,2016,2016-10-31,1610612737,1610612758,1,2.0,0,2.0,1.00,1.000000
3,2016,2016-11-02,1610612737,1610612747,0,2.0,0,3.0,1.00,1.000000
4,2016,2016-11-04,1610612737,1610612764,0,2.0,0,-1.0,0.75,0.750000
5,2016,2016-11-05,1610612737,1610612745,1,1.0,1,-2.0,0.60,0.600000
6,2016,2016-11-08,1610612737,1610612739,1,3.0,0,1.0,0.60,0.666667
7,2016,2016-11-09,1610612737,1610612741,1,1.0,1,2.0,0.60,0.714286
8,2016,2016-11-12,1610612737,1610612755,1,3.0,0,3.0,0.60,0.750000
9,2016,2016-11-15,1610612737,1610612748,1,3.0,0,4.0,0.80,0.777778


## Step 2 — Self-join to get opponent's per-team features, then diff

Each row's "opponent values" live in a *different* row of the same DataFrame (the opposite team's perspective on the same game). We rebuild them via a self-join on `(GameID, OppID) → (GameID, TeamID)`, then compute `Diff_* = team − opp`.

In [4]:
PER_TEAM = ['Days_Rest', 'Is_B2B', 'Win_Streak', 'Last5_Win_Pct', 'Last10_Win_Pct']

opp_lookup = (mm[['GameID', 'TeamID'] + PER_TEAM]
              .rename(columns={'TeamID': 'OppID',
                               **{c: f'Opp_{c}' for c in PER_TEAM}}))

mm = mm.merge(opp_lookup, on=['GameID', 'OppID'], how='left')

# Diff versions of the continuous per-team features
mm['Diff_Days_Rest']      = mm['Days_Rest']      - mm['Opp_Days_Rest']
mm['Diff_Win_Streak']     = mm['Win_Streak']     - mm['Opp_Win_Streak']
mm['Diff_Last5_Win_Pct']  = mm['Last5_Win_Pct']  - mm['Opp_Last5_Win_Pct']
mm['Diff_Last10_Win_Pct'] = mm['Last10_Win_Pct'] - mm['Opp_Last10_Win_Pct']

# B2B is binary -- keep both perspectives separately
mm = mm.rename(columns={'Is_B2B': 'Team_Is_B2B'})

# Drop the per-team intermediates (we keep the diffs and the B2B flags)
mm = mm.drop(columns=['Days_Rest', 'Win_Streak', 'Last5_Win_Pct', 'Last10_Win_Pct',
                      'Opp_Days_Rest', 'Opp_Win_Streak',
                      'Opp_Last5_Win_Pct', 'Opp_Last10_Win_Pct'])

mm[['TeamID','OppID','Team_Is_B2B','Opp_Is_B2B',
    'Diff_Days_Rest','Diff_Win_Streak',
    'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct']].head(10)

,TeamID,OppID,Team_Is_B2B,Opp_Is_B2B,Diff_Days_Rest,Diff_Win_Streak,Diff_Last5_Win_Pct,Diff_Last10_Win_Pct
0,1610612737,1610612764,<NA>,<NA>,NaN,NaN,NaN,NaN
1,1610612737,1610612755,0,0,-1.0,2.0,1.000000,1.000000
2,1610612737,1610612758,0,0,0.0,1.0,0.333333,0.333333
3,1610612737,1610612747,0,1,1.0,6.0,0.750000,0.750000
4,1610612737,1610612764,0,0,0.0,2.0,0.750000,0.750000
5,1610612737,1610612745,1,0,-2.0,-3.0,0.000000,0.000000
6,1610612737,1610612739,0,0,0.0,-5.0,-0.400000,-0.333333
7,1610612737,1610612741,1,0,-1.0,1.0,0.200000,0.142857
8,1610612737,1610612755,0,1,2.0,2.0,0.400000,0.625000
9,1610612737,1610612748,0,1,2.0,9.0,0.800000,0.555556


## Step 3 — Head-to-head this season

Each row already has `TeamID` and `OppID`, so we can compute this team's prior win pct vs that specific opponent within the season directly. NaN means it's the first meeting this season.

In [5]:
def add_h2h(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['Season','TeamID','OppID','DayDate','GameID']).reset_index(drop=True).copy()
    grp = df.groupby(['Season','TeamID','OppID'], sort=False)
    cum_wins  = grp['Target_Win'].cumsum() - df['Target_Win']   # wins BEFORE current
    cum_games = grp.cumcount()                                  # games BEFORE current
    df['H2H_Win_Pct'] = cum_wins / cum_games.replace(0, np.nan)
    return df


mm = add_h2h(mm)
mm[['Season','DayDate','TeamID','OppID','Target_Win','H2H_Win_Pct']].head(15)

,Season,DayDate,TeamID,OppID,Target_Win,H2H_Win_Pct
0,2016,2017-01-13,1610612737,1610612738,0,NaN
1,2016,2017-02-27,1610612737,1610612738,1,0.000000
2,2016,2017-04-06,1610612737,1610612738,1,0.500000
3,2016,2016-11-08,1610612737,1610612739,1,NaN
4,2016,2017-03-03,1610612737,1610612739,0,1.000000
5,2016,2017-04-07,1610612737,1610612739,1,0.500000
6,2016,2017-04-09,1610612737,1610612739,1,0.666667
7,2016,2016-11-22,1610612737,1610612740,0,NaN
8,2016,2017-01-05,1610612737,1610612740,1,0.000000
9,2016,2016-11-09,1610612737,1610612741,1,NaN


## Step 4 — Elo-implied win probability

`Diff_Pregame_Elo` already encodes `team_elo − opp_elo` from this team's view (computed in Step 0). Add the home-court bump (positive for the home team, negative for the away team) and convert with the standard logistic:

$$P(\text{this team wins}) = \frac{1}{1 + 10^{-(\text{Diff\_Pregame\_Elo}\,+\,\text{HCA}_{\text{signed}}) / 400}}$$

where $\text{HCA}_{\text{signed}}$ is +100 if `Is_Home` and −100 otherwise.

In [6]:
hca = np.where(mm['Is_Home'].astype(bool), ELO_HOME_ADV, -ELO_HOME_ADV)
mm['Elo_Win_Prob'] = 1.0 / (1.0 + 10 ** (-((mm['Diff_Pregame_Elo'] + hca)) / 400.0))

print(f"Elo_Win_Prob range: [{mm['Elo_Win_Prob'].min():.3f}, "
      f"{mm['Elo_Win_Prob'].max():.3f}]")
print(f"Home rows mean: {mm.loc[mm['Is_Home']==1, 'Elo_Win_Prob'].mean():.3f}  "
      f"(should be > 0.5)")
print(f"Away rows mean: {mm.loc[mm['Is_Home']==0, 'Elo_Win_Prob'].mean():.3f}  "
      f"(should be < 0.5)")

Elo_Win_Prob range: [0.030, 0.970]
Home rows mean: 0.624  (should be > 0.5)
Away rows mean: 0.376  (should be < 0.5)


## Step 5 — Save `matchups_v2.csv`

In [7]:
mm = mm.sort_values(['Season','DayDate','GameID','TeamID']).reset_index(drop=True)

mm.to_csv(OUT_DIR / OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV}  ({len(mm):,} rows, {len(mm.columns)} cols)")

new_cols = ['Team_Is_B2B','Opp_Is_B2B',
            'Diff_Days_Rest','Diff_Win_Streak',
            'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct',
            'H2H_Win_Pct','Elo_Win_Prob']
print(f"\nNew features added ({len(new_cols)}):")
for c in new_cols:
    null_count = mm[c].isna().sum()
    print(f"  {c:<24s}  (NaN: {null_count})")

Wrote matchups_v2.csv  (23,938 rows, 130 cols)

New features added (8):
  Team_Is_B2B               (NaN: 300)
  Opp_Is_B2B                (NaN: 300)
  Diff_Days_Rest            (NaN: 318)
  Diff_Win_Streak           (NaN: 318)
  Diff_Last5_Win_Pct        (NaN: 318)
  Diff_Last10_Win_Pct       (NaN: 318)
  H2H_Win_Pct               (NaN: 8686)
  Elo_Win_Prob              (NaN: 0)


## Sanity checks

Same checks as the original notebook — the team-perspective format gives us strong consistency checks:

- `Diff_*` features → **sum = 0** across both perspectives of a game (team's diff = −opp's diff)
- `Elo_Win_Prob` → **sum = 1** (one team's win prob + the other's = 1)
- `H2H_Win_Pct` → sum = 1 when both perspectives are non-NaN

In [8]:
# Check 1: Last5_Win_Pct correctness via independent reconstruction
prior_count = mm.groupby(['TeamID','Season']).cumcount()
sample_idx = mm[(prior_count >= 5) & mm['Diff_Last5_Win_Pct'].notna()].index[0]
sample = mm.loc[sample_idx]

team_priors = mm[(mm['TeamID']==sample['TeamID']) & (mm['Season']==sample['Season']) &
                 (mm['DayDate']<sample['DayDate'])].sort_values('DayDate').tail(5)
opp_priors  = mm[(mm['TeamID']==sample['OppID'])  & (mm['Season']==sample['Season']) &
                 (mm['DayDate']<sample['DayDate'])].sort_values('DayDate').tail(5)
diff_manual = team_priors['Target_Win'].mean() - opp_priors['Target_Win'].mean()

print(f"Sample team={sample['TeamID']}, opp={sample['OppID']}, "
      f"season={sample['Season']}, date={sample['DayDate'].date()}")
print(f"  Team last5 win pct: {team_priors['Target_Win'].mean():.3f}")
print(f"  Opp  last5 win pct: {opp_priors['Target_Win'].mean():.3f}")
print(f"  Manual diff:        {diff_manual:.3f}")
print(f"  File Diff_Last5:    {sample['Diff_Last5_Win_Pct']:.3f}")
print(f"  Match: {np.isclose(diff_manual, sample['Diff_Last5_Win_Pct'])}")

# Check 2: Diff_* should sum to 0 across both perspectives of the same game
print("\n=== Diff sums per game (expected ~0) ===")
for col in ['Diff_Days_Rest','Diff_Win_Streak',
            'Diff_Last5_Win_Pct','Diff_Last10_Win_Pct']:
    s = mm.groupby('GameID')[col].sum(min_count=2)   # NaN if either side is NaN
    print(f"  {col:<22s}  mean abs sum: {s.abs().mean():.6f}")

# Check 3: Elo_Win_Prob should sum to 1.0 across both perspectives
prob_sum = mm.groupby('GameID')['Elo_Win_Prob'].sum()
print(f"\nElo_Win_Prob per-game sum: mean={prob_sum.mean():.6f}, "
      f"std={prob_sum.std():.2e}  (expect 1.0, ~0)")

# Check 4: H2H_Win_Pct should sum to 1.0 when both perspectives non-NaN
h2h_sum = mm.groupby('GameID')['H2H_Win_Pct'].sum(min_count=2)
nonnan = h2h_sum.dropna()
print(f"H2H_Win_Pct per-game sum (when both non-NaN): "
      f"mean={nonnan.mean():.6f}, n={len(nonnan)}  (expect 1.0)")

Sample team=1610612758, opp=1610612753, season=2016, date=2016-11-03
  Team last5 win pct: 0.400
  Opp  last5 win pct: 0.250
  Manual diff:        0.150
  File Diff_Last5:    0.150
  Match: True

=== Diff sums per game (expected ~0) ===
  Diff_Days_Rest          mean abs sum: 0.000000
  Diff_Win_Streak         mean abs sum: 0.000000
  Diff_Last5_Win_Pct      mean abs sum: 0.000000
  Diff_Last10_Win_Pct     mean abs sum: 0.000000

Elo_Win_Prob per-game sum: mean=1.000000, std=3.46e-17  (expect 1.0, ~0)
H2H_Win_Pct per-game sum (when both non-NaN): mean=1.000000, n=7626  (expect 1.0)


## Plug into your model

When you load `matchups_v2.csv`, identify the feature columns dynamically (any `Diff_*` plus the per-team binary/probability features). Drop the rows from each team's first ~1 game of the season where rolling features are NaN.

```python
df = pd.read_csv("matchups_v2.csv", parse_dates=['DayDate'])
df = df.dropna(subset=['Diff_Avg_Score']).copy()

feature_cols = [c for c in df.columns if c.startswith('Diff_')] + [
    'Is_Home', 'Team_Is_B2B', 'Opp_Is_B2B',
    'H2H_Win_Pct', 'Elo_Win_Prob',
]
X = df[feature_cols]
y = df['Target_Win']
```

Your existing Leave-One-Season-Out CV loop works unchanged — both rows of any given game are always in the same season, so there's no train/val leak.

In [12]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, brier_score_loss

# 2. Initialize Arrays to Store Predictions
oof_preds = np.zeros(len(df))        # Out-of-fold predictions (probabilities)
oof_preds_class = np.zeros(len(df))  # Out-of-fold predictions (binary class)

# 3. Define CatBoost Parameters
# Notice how the parameter names change slightly from XGBoost!
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'depth': 4,                   # CatBoost's equivalent to max_depth
    'learning_rate': 0.03,
    'iterations': 150,            # CatBoost's equivalent to n_estimators
    'subsample': 0.8,
    'colsample_bylevel': 0.8,     # CatBoost's equivalent to colsample_bytree
    'random_seed': 42,            # Equivalent to random_state
    'verbose': False              # Keeps the output perfectly clean
}

unique_seasons = sorted(df['Season'].unique())
models = {} # Store models for feature importance later

print("\nStarting Leave-One-Season-Out CV with CatBoost...\n")

for val_season in unique_seasons:
    # --- Split Data ---
    train_idx = df['Season'] != val_season
    val_idx   = df['Season'] == val_season
    
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val     = X[val_idx], y[val_idx]
    
    # --- Initialize and Train Model ---
    clf = CatBoostClassifier(**cb_params)
    
    # Using early stopping to prevent overfitting
    # CatBoost makes this super easy with early_stopping_rounds
    clf.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=20 # Stops if eval metric doesn't improve for 20 trees
    )
    
    # --- Predict on Validation Season ---
    # predict_proba returns [prob_loss, prob_win]
    val_probs = clf.predict_proba(X_val)[:, 1]
    
    # CatBoost has a built-in .predict() that returns the hard class directly!
    val_preds = clf.predict(X_val) 
    
    # --- Store Results ---
    oof_preds[val_idx] = val_probs
    oof_preds_class[val_idx] = val_preds
    models[val_season] = clf
    
    # --- Evaluate this specific season ---
    season_acc = accuracy_score(y_val, val_preds)
    season_auc = roc_auc_score(y_val, val_probs)
    season_brier = brier_score_loss(y_val, val_probs)
    print(f"Held out Season {val_season} | Test Games: {len(y_val)} | Acc: {season_acc:.4f} | AUC: {season_auc:.4f} | Brier: {season_brier:.4f}")

# --- Overall Evaluation ---
overall_acc = accuracy_score(y, oof_preds_class)
overall_auc = roc_auc_score(y, oof_preds)
overall_logloss = log_loss(y, oof_preds)
overall_brier = brier_score_loss(y, oof_preds)

print("\n" + "="*35)
print("CATBOOST LOSO CV RESULTS")
print("="*35)
print(f"Overall Accuracy : {overall_acc:.4f}")
print(f"Overall ROC AUC  : {overall_auc:.4f}")
print(f"Overall Log Loss : {overall_logloss:.4f}")
print(f"Overall Brier    : {overall_brier:.5f}")
print("="*35)


Starting Leave-One-Season-Out CV with CatBoost...

Held out Season 2016 | Test Games: 2428 | Acc: 0.6376 | AUC: 0.6935 | Brier: 0.2213
Held out Season 2017 | Test Games: 2428 | Acc: 0.6483 | AUC: 0.7121 | Brier: 0.2160
Held out Season 2018 | Test Games: 2428 | Acc: 0.6590 | AUC: 0.7200 | Brier: 0.2137
Held out Season 2019 | Test Games: 2086 | Acc: 0.6577 | AUC: 0.7098 | Brier: 0.2171
Held out Season 2020 | Test Games: 2128 | Acc: 0.6217 | AUC: 0.6745 | Brier: 0.2271
Held out Season 2021 | Test Games: 2428 | Acc: 0.6376 | AUC: 0.6934 | Brier: 0.2220
Held out Season 2022 | Test Games: 2428 | Acc: 0.6289 | AUC: 0.6736 | Brier: 0.2271
Held out Season 2023 | Test Games: 2430 | Acc: 0.6481 | AUC: 0.7210 | Brier: 0.2131
Held out Season 2024 | Test Games: 2418 | Acc: 0.6634 | AUC: 0.7250 | Brier: 0.2121
Held out Season 2025 | Test Games: 2418 | Acc: 0.6935 | AUC: 0.7463 | Brier: 0.2055

CATBOOST LOSO CV RESULTS
Overall Accuracy : 0.6498
Overall ROC AUC  : 0.7078
Overall Log Loss : 0.6243
Over